# Lab 10: Large Language Models (LLMs)

**Week 3, Day 1: Time Series and Natural Language Processing** adapted by Prof. Nils Murrugarra from Univ of Pittsburgh.

**By Neuromatch Academy**

__Content creators:__ Lyle Ungar, Jordan Matelsky, Konrad Kording, Shaonan Wang, Alish Dipani

__Content reviewers:__ Shaonan Wang, Weizhe Yuan, Dalia Nasr, Stephen Kiilu, Alish Dipani, Dora Zhiyu Yang, Adrita Das

__Content editors:__ Konrad Kording, Shaonan Wang

__Production editors:__ Konrad Kording, Spiros Chavlis, Konstantine Tsafatinos

---
# Tutorial Objectives

This tutorial provides a comprehensive overview of modern natural language processing (NLP). It introduces GPT, along with a detailed exploration of the underlying NLP pipeline. Participants will learn about the core concepts, functionalities, and applications of these architectures, as well as gain insights into prompt engineering and the current and future developments of GPT.

---
# Setup

In [1]:
# @title Install dependencies
# @markdown **WARNING**: There may be *errors* and/or *warnings* reported during the installation. However, they are to be ignored.
!pip install typing_extensions --quiet
!pip install accelerate --quiet
!pip install datasets --quiet
!pip install evaluate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [2]:
!pip install datasets==3.6.0 # new: nm # Solution: https://github.com/stanfordnlp/dspy/issues/8590

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
# @title Install and import feedback gadget

!pip3 install vibecheck datatops --quiet

from vibecheck import DatatopsContentReviewContainer
def content_review(notebook_section: str):
    return DatatopsContentReviewContainer(
        "",  # No text prompt
        notebook_section,
        {
            "url": "https://pmyvdlilci.execute-api.us-east-1.amazonaws.com/klab",
            "name": "neuromatch_dl",
            "user_key": "f379rz8y",
        },
    ).render()


feedback_prefix = "W3D1_T2"

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.4 MB/s eta 0:00:00


In [4]:
# Imports
import random
import numpy as np
from typing import Iterable, List
from tqdm.notebook import tqdm
from typing import Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tokenizers import Tokenizer, Regex, models, normalizers, pre_tokenizers, trainers, processors

In [5]:
# @title Set random seed

# @markdown Executing `set_seed(seed=seed)` you are setting the seed

# for DL its critical to set the random seed so that students can have a
# baseline to compare their results to expected results.
# Read more here: https://pytorch.org/docs/stable/notes/randomness.html

# Call `set_seed` function in the exercises to ensure reproducibility.
import random
import numpy as np

def set_seed(seed=None):
  if seed is None:
    seed = np.random.choice(2 ** 32)
  random.seed(seed)
  np.random.seed(seed)
  print(f'Random seed {seed} has been set.')


set_seed(seed=2023)  # change 2023 with any number you like

Random seed 2023 has been set.


In [6]:
# @title Set device (GPU or CPU). Execute `set_device()`

# Inform the user if the notebook uses GPU or CPU.

def set_device():
  """
  Set the device. CUDA if available, CPU otherwise

  Args:
    None

  Returns:
    Nothing
  """
  device = "cuda" if torch.cuda.is_available() else "cpu"
  if device != "cuda":
    print("WARNING: For this notebook to perform best, "
        "if possible, in the menu under `Runtime` -> "
        "`Change runtime type.`  select `GPU` ")
  else:
    print("GPU is enabled in this notebook.")

  return device

In [7]:
DEVICE = set_device()
SEED = 2021
set_seed(seed=SEED)

GPU is enabled in this notebook.
Random seed 2021 has been set.


# Section 1: NLG with GPT

In this section we will learn about Natural Language Generation with Generative Pretrained Transformers.

## Learning goals
* How to produce language with GPTs

## Using state-of-the-art (SOTA) Models

Unless you are writing your own experimental DL research (and sometimes even then!) it is _far_ more common these days to use the HuggingFace model library to import and start working with state-of-the-art models quickly. In this section, we will show you how to do that.

We will download a pretrained model from the hf `transformers` library that is used to generate text. We will then fine-tune it on a different dataset, using the `hf.datasets` library and the HuggingFace Trainer classes to make the process as easy as possible, and we'll see that we can accomplish all of this in just a few lines of easily maintained code.

Ultimately, we will have a _working_ generator... for code!

We're first going to pick a tokenizer. You can see some of the options [here](https://huggingface.co/transformers/pretrained_models.html). We'll use CodeParrot tokenizer, which is a BPE tokenizer. But you can choose (or build!) another if you'd like to try offroading!

In [8]:
from transformers import AutoTokenizer
from datasets import load_dataset

In [9]:
# print(datasets.__version__)

In [10]:
# tokenizer = AutoTokenizer.from_pretrained("codeparrot/codeparrot-small")

Next, we'll download a pre-built model architecture. CodeParrot (the model) is a GPT-2 model, which is a transformer-based language model. You can see some of the options [here](https://huggingface.co/transformers/pretrained_models.html). But you can choose (or build!) another!

Note that `codeparrot/codeparrot` (https://huggingface.co/codeparrot/codeparrot) is about 7GB to download (so it may take a while, or it may be too large for your runtime if you're on a free Colab). Instead, we will use a smaller model, `codeparrot/codeparrot-small` (https://huggingface.co/codeparrot/codeparrot-small), which is only ~500MB.

To run everything together — tokenization, model, and de-tokenization, we can use the `pipeline` function from `transformers`.

In [11]:
from transformers import AutoModelForCausalLM
from transformers import pipeline

model = AutoModelForCausalLM.from_pretrained("codeparrot/codeparrot-small") # TODO: Try with default model
tokenizer = AutoTokenizer.from_pretrained("codeparrot/codeparrot-small") # new nm
generation_pipeline = pipeline(
    "text-generation", # The task to run. This tells hf what the pipeline steps are
    model=model, # The model to use; can also pass the string here;
    tokenizer=tokenizer, # The tokenizer to use; can also pass the string name here.
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/903 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/457M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
input_prompt = '''\
def simple_add(a: int, b: int) -> int:
    """
    Adds two numbers together and returns the result.
    """'''

# Return tensors for PyTorch:
inputs = tokenizer(input_prompt, return_tensors="pt")

Recall that these tokens are integer indices in the vocabulary of the tokenizer. We can use the tokenizer to decode these tokens into a string, which we can print out to see what the model generates.

In [13]:
input_token_ids = inputs["input_ids"]
input_strs = tokenizer.convert_ids_to_tokens(*input_token_ids.tolist())

print(*zip(input_strs, input_token_ids[0]))

('def', tensor(318)) ('Ġsimple', tensor(3486)) ('_', tensor(63)) ('add', tensor(525)) ('(', tensor(8)) ('a', tensor(65)) (':', tensor(26)) ('Ġint', tensor(1109)) (',', tensor(12)) ('Ġb', tensor(330)) (':', tensor(26)) ('Ġint', tensor(1109)) (')', tensor(9)) ('Ġ->', tensor(1035)) ('Ġint', tensor(1109)) (':', tensor(26)) ('ĊĠĠĠ', tensor(272)) ('Ġ"""', tensor(408)) ('ĊĠĠĠ', tensor(272)) ('ĠAdds', tensor(15747)) ('Ġtwo', tensor(2877)) ('Ġnumbers', tensor(5579)) ('Ġtogether', tensor(10451)) ('Ġand', tensor(436)) ('Ġreturns', tensor(2529)) ('Ġthe', tensor(314)) ('Ġresult', tensor(754)) ('.', tensor(14)) ('ĊĠĠĠ', tensor(272)) ('Ġ"""', tensor(408))


**(Quick knowledge-check: what are the weirdly-rendering characters representing?)**

This model is already ready to use! Let's give it a try. (Note that we don't use `inputs` — we just generated that to show the initial tokenization steps.)

Here, we use the `pipeline` we created earlier to combine all our components. If you were writing a Copilot-style code-completer, you could get away with wrapping this single line in a nice API and calling it a day!

Play with the hyperparameters and see what kinds of outputs you can get. Temperature measures how much randomness is added to the model's predictions. Higher temperature means more randomness and lower temperature means less randomness. More randomness in the latent space will lead to wilder predictions and potentially more creative answers. A good place to start is `0.2`. You can also try changing the `max_length` parameter, which controls how long the generated code can be (though the model can opt to put a "stop" token in the middle of the sequence, so it may not always generate exactly this many tokens).

In [14]:
# outputs = generation_pipeline(input_prompt, max_length=100, num_return_sequences=1, temperature=0.2, truncation=True)
# outputs = generation_pipeline(input_prompt, max_length=100, num_return_sequences=1, temperature=0.2) # new
outputs = generation_pipeline(input_prompt, max_length=100, num_return_sequences=1, temperature=0.2, truncation=True) # new

Passing `generation_config` together with generation-related arguments=({'temperature', 'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [15]:
print(outputs[0]["generated_text"])

def simple_add(a: int, b: int) -> int:
    """
    Adds two numbers together and returns the result.
    """
    return a + b


def simple_sub(a: int, b: int) -> int:
    """
    Subtracts two numbers together and returns the result.
    """
    return a - b


def simple_mul(a: int, b: int) -> int:
    """
    Multiplies two numbers together and returns the result.
    """
    return a * b


def simple_div(a: int, b: int) -> int:
    """
    Divides two numbers together and returns the result.
    """
    return a / b


def simple_mod(a: int, b: int) -> int:
    """
    Mines two numbers together and returns the result.
    """
    return a % b


def simple_mul_with_args(a: int, b: int, *args: Any, **kwargs: Any) -> int:
    """
    Mines two numbers together and adds the result.
    """
    a = simple_add(a, b, a)
    a = simple_sub(a, b, a, b)
    a = simple_mod(a, args, kwargs)
    return a


def main() ->


Let's see if we can fool our model now! The huggingface documentation tells us that the codeparrot model was trained to generate Python code ([docs](https://huggingface.co/codeparrot/codeparrot-small)). Let's see if we can get it to generate some JavaScript.

In [16]:
input_prompt = "class SimpleAdder {"

print(generation_pipeline(input_prompt, max_length=100, num_return_sequences=1, temperature=0.2)[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


class SimpleAdder {
    'name': 'Simple Adder',
    'type': 'object',
    'properties': {
        'name': {'type':'string'},
        'type': {'type':'string'},
        'description': {'type':'string'},
        'id': {'type': 'integer'},
        'type': {'type':'string'},
        'type': {'type':'string'},
        'location': {'type':'string'},
        'type': {'type':'string'},
       'start_date': {'type':'string'},
        'end_date': {'type':'string'},
        'end_date_by_year': {'type':'string'},
       'start_date_by_month': {'type':'string'},
       'start_date_by_day': {'type':'string'},
       'start_date_by_year_month':'string',
        'end_date_by_year_day':'string',
        'end_date_by_month_year':'string',
        'end_end_date': {'type':'string'},
        'end_date_by_year':'string',
       'start_start': {'type': 'integer


Yikes! I don't know what it generated for you, but what it made for me was:

```python
class SimpleAdder {
    public:
        class SimpleAdder(object):
            def __init__(self, a, b):
                self.a = a
                self.b = b

            def __call__(self, x):
                return self.a + x
```

**Ew!** That's wrong in a _lot_ of ways. But it's understandable: Our model can't really generalize outside of the domain in which it was trained. And so probably there were a few Python files that included syntax of other languages (perhaps generators for other code?). So the model knows that there's some mysterious syntax that uses curly brackets... But it's not sure about anything else. (For the programming-language hobbyists among you: The `public` notation looks to me a lot like the model is trying to do something C-flavored and perhaps something Java-flavored; I like it! But it's definitely not JavaScript.)

What are the major observations?

* The syntax it's generating rapidly and devolves into Python; it can predict only a few characters of non-Python before falling back into its familiar training territory.
* The part of the code that follows Python syntax is valid and resembles a useful class definition (although if you look closely, it doesn't seem to do anything useful with the `b` attribute...). This tells us that the model "understands" its problem domain but hasn't been trained on the correct data to solve our new problem.

## Fine-Tuning

Alright, so we have a model that can generate code. But now, we want to fine-tune it to generate JavaScript.

Assuming the data will be too large to fit on disk on Colab, we'll use the `load_dataset` function to download only part of the dataset. There's a JavaScript subset to the codeparrot dataset, which we'll use as an example… But you can use any dataset you like! We recommend filtering datasets by task category (e.g., text generation) to get the most relevant datasets. Still, you can use any dataset you like if you can configure the data loader to use it. (Consider, for example, [this one](https://huggingface.co/datasets/angie-chen55/javascript-github-code).)

> **Choose a dataset from the [HuggingFace datasets library](https://huggingface.co/datasets?task_categories=task_categories:text-generation&sort=downloads).**

In [17]:
# Unlike _some_ code-generator models on the market, we'll limit our training data by license :)
dataset = load_dataset(
    "codeparrot/github-code",
    streaming=True,
    split="train",
    languages=["JavaScript"],
    licenses=["mit", "isc", "apache-2.0"],
    trust_remote_code=True,
)
# Print the schema of the first example from the training set:
print({k: type(v) for k, v in next(iter(dataset)).items()})

README.md: 0.00B [00:00, ?B/s]

github-code.py: 0.00B [00:00, ?B/s]

{'code': <class 'str'>, 'repo_name': <class 'str'>, 'path': <class 'str'>, 'language': <class 'str'>, 'license': <class 'str'>, 'size': <class 'int'>}


Like training any model, we need to define a training loop and an evaluation metric.

This is made overwhelmingly easy with the `transformers` library. Specifically, look below at all of the code you can avoid using the huggingface infrastructure. (In the past, we've used PyTorch Lightning, which had a similar training-loop abstraction. Do you have preferences between these two libraries?)

### Implement the code to fine-tune the model

Here are the big pieces of what we do below:

* **Create a `TrainingArguments` object.** This serializable object (i.e., you can save it to memory or disk) makes it easy to train a model reproducibly with the same hyperparameters (this certainly beats having a bunch of global variables in your notebook!).
* **Encode the dataset.** This is effectively just passing everything through the tokenizer, with a padding step that fills the end of each sequence with the padding token.
* **Define our metrics.** We use the `accuracy` metric here (look at the 4th line in the code cell).
* **Create a data collator.** This function takes a list of examples and returns a batch of examples. The `DataCollatorForLanguageModeling` class is a convenient way to do this.
* **Create a `Trainer` object.** This class wraps the training loop and makes it easy to train a model. It's a bit like the `Trainer` class in PyTorch Lightning, but it's a bit more flexible and works with non-PyTorch models as well.

In [18]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from evaluate import load
metric = load("accuracy")

# Trainer:
training_args = TrainingArguments(
    output_dir="./codeparrot",
    max_steps=100, # orig: 100 [student], new: 1000 (1 epoch) [student], 3000 (1 epoch)[instructor]
    per_device_train_batch_size=1,
    report_to="none",
    # num_train_epochs=10,
)

tokenizer.pad_token = tokenizer.eos_token

encoded_dataset = dataset.map(
    lambda x: tokenizer(x["code"], truncation=True, padding="max_length"),
    batched=True,
    remove_columns=["code"],
)


# Metrics for loss:
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis=-1)
  return metric.compute(predictions=predictions, references=labels)


# Data collator:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False,
)

# Trainer:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

In [19]:
# Run the actual training:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 0, 'bos_token_id': 0, 'pad_token_id': 0}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=2.6775494384765626, metrics={'train_runtime': 48.6462, 'train_samples_per_second': 2.056, 'train_steps_per_second': 2.056, 'total_flos': 67720288665600.0, 'train_loss': 2.6775494384765626, 'epoch': 1.0})

### Coding Exercise 1.1: Implement the code to generate text after fine-tuning.

To generate text, we provide input tokens to the model, let it generate the next token and append it into the input tokens. Now, keep repeating this process until you reach the desired output length.

```python
def generate_text(model, input_prompt):
  # Number of tokens to generate
  num_tokens = 100

  # Move the model to the CPU for inference
  model.to("cpu")

  # Print input prompt
  print(f'Input prompt: \n{input_prompt}')

  #################################################
  # Implement a the correct tokens and outputs
  raise NotImplementedError("Text Generation")
  #################################################

  # Encode the input prompt
  # https://huggingface.co/docs/transformers/en/main_classes/tokenizer
  input_tokens = ...

  # Turn off storing gradients
  with torch.no_grad():
    # Keep iterating until num_tokens are generated
    for tkn_idx in tqdm(range(num_tokens)):
      # Forward pass through the model
      # The model expects the tensor to be of Long or Int dtype
      output = ...
      # Get output logits
      logits = output.logits[-1, :]
      # Convert into probabilities
      probs = nn.functional.softmax(logits, dim=-1)
      # Get the index of top token
      top_token = ...
      # Append the token into the input sequence
      input_tokens.append(top_token)

  # Decode and print the generated text
  # https://huggingface.co/docs/transformers/en/main_classes/tokenizer
  decoded_text = ...
  return decoded_text

# print(f'Generated text: \n{generate_text(model, input_prompt)}')

```

# Activity 10
Please, develop the previous coding exercise. Then, copy your output and submit on [AhaSlides](https://ahaslides.com/XAIT2).

In [ ]:
# TODO: add code here

Of course, your results will be slightly different. Here's what I got:

```javascript
class SimpleAdder {
    constructor(a, b) {
        this.a = a;
        this.b = b;
    }

    add(
```

Much better! The model is no longer generating Python code, and it's not trying to jam Python-flavored syntax into other languages. It's still imperfect, but it's much better than before! (And, of course, remember that this is just a small model, and we didn't train it for very long. You can either try training it for longer or using a larger model to get better results.)

---
# Section 2: GPT Today and Tomorrow

Limitation of the current models.

## Play around with LLMs

1. Try using LLMs' API to do tasks, such as utilizing the GPT-2 API to extend text from a provided context. To achieve this, ensure you have a HuggingFace account and secure an API token.

In [20]:
# Ref: https://huggingface.co/docs/transformers/model_doc/gpt2

import torch
from transformers import pipeline

pipeline = pipeline(task="text-generation", model="openai-community/gpt2", dtype=torch.float16, device=0)
x = pipeline("Hello, I'm a language model")
# x = pipeline("The goal of life is")
output_text = x[0]['generated_text']
print(f'output: {output_text}')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


output: Hello, I'm a language modeler. I also write code to create and consume user interfaces, such as your browser's user interface. I also try to understand how our UI works. I also write complex, complex, complex applications. The more complex the UI, the more complex the UI becomes.

This is the type of work you do for an employer. I have no idea how you got started. This is all part of my job.

The next time you meet a new boss, or you meet someone new who is new to your profession, I want to know about your experience and how you got through it. I want to share my experiences with you. If you want to share your experience, please do so here.

Thank you,

Dan


---
# Summary

In this tutorial you have become familiar with modern natural language processing (NLP) architectures. We learned about the core concepts, functionalities, and applications of these architectures. We also gain insights into prompt engineering and we learned about GPT.

---
# Acknowledgement
Material is based on [Neuromatch](https://github.com/NeuromatchAcademy/course-content-dl/tree/main/tutorials)